# Tests des fonctions de calcul de la saturation

In [1]:
import json
from datetime import date, timedelta

import pandas as pd
from pandas import NamedAgg

#from saturation_image_quali_prod import (
from saturation_image_quali import (
    filter_sessions_duration,
    get_sampled_state_poc,
    to_sampled_state_grp,
    to_state_grp_d,
    to_state_poc_d,
)

SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

#day = date(2026,7, 5)
day = date(2026,7, 14)
#date_calcul = "2026-07-05"
date_calcul = "2026-07-14"
date_file = date_calcul.replace("-", "")

data_quali = "../data/"

In [2]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for POC and stations."""
    e5_str = pd.read_csv("../data_DMR_e2_e3/e5_05-07-2026.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]


In [3]:
sessions_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/sessions/production.parquet", engine="pyarrow")
statuses_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/statuses/production.parquet", engine="pyarrow")

In [4]:
sessions = pd.read_csv('../data_test/donnees_sessions_FRHPCPNF050462_05-07-2026.csv')[['start', 'end', 'id_pdc_itinerance']]
sessions['start'] = pd.to_datetime(sessions['start'])
sessions['end'] = pd.to_datetime(sessions['end'])
statuses = pd.DataFrame({ID_POC:[], "horodatage":[], "etat_pdc":[], "occupation_pdc":[]})
statuses['horodatage'] = pd.to_datetime(statuses['horodatage'], utc=True)
# statuses = pd.DataFrame()
statics = pd.DataFrame({ID_POC:['FRHPCENF050462001', 'FRHPCENF050462002', 'FRHPCENF050462004' ], ID_STATION:['FRHPCPNF050462']*3})

samples_per_day = SAMPLES #72 #288

e2_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][0]
e2_pdc = pd.DataFrame(json.loads(e2_str))

e3_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][1]
e3_station = pd.DataFrame(json.loads(e3_str))

e5_statics = read_statics(day, MIN_POWER)

# FRHPCENF050462001
e2_pdc = e2_pdc[e2_pdc[ID_POC].str[:14] == 'FRHPCENF050462']
e3_station = e3_station[e3_station[ID_STATION] == 'FRHPCPNF050462']


In [5]:
#e5_statics[e5_statics[ID_STATION] == 'FRHPCPNF080266TOTEM']
#e5_statics

## test local

In [6]:
samples_per_day = 288
#sampled_state_poc = get_sampled_state_poc(day, samples_per_day, sessions, statuses)

In [7]:
#sampled_state_poc[sampled_state_poc[ID_POC] == 'FRHPCENF050462001']

In [8]:
#state_poc_d = to_state_poc_d(sampled_state_poc, samples_per_day)

In [9]:
#state_poc_d, e2_pdc

In [10]:
#sample_state_station = to_sampled_state_grp(sampled_state_poc, statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
#state_station_h = to_state_grp_h(sample_state_station, ID_STATION, SAMPLES, SATURE_H)
#state_station_d = to_state_grp_d(state_station_h, ID_STATION)

In [11]:
#state_station_d

In [12]:
#e3_station

## test global

In [13]:
samples_per_day = 288

min_duration = timedelta(minutes=24 * 60 / samples_per_day)
max_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)
sessions = filter_sessions_duration(sessions_s3, min_duration=min_duration, max_duration=max_duration)
sessions_poc = sessions.groupby(ID_POC).agg(
                sessions_nb=NamedAgg("energy", "count"),
                energy_cum=NamedAgg("energy", "sum")
            ).reset_index()


In [14]:
e5_statics[[ID_POC, ID_STATION]]
sessions_stations = pd.merge(e5_statics[[ID_POC, ID_STATION]], sessions_poc, on=ID_POC, how='left').fillna(0)
info_sessions_stations = sessions_stations[[ID_STATION, 'sessions_nb', 'energy_cum']].groupby(ID_STATION).sum().reset_index()
info_sessions_stations


,id_station_itinerance,sessions_nb,energy_cum
0,FRALDPFR00916,1,56.7
1,FRALDPFR00950,2,93.587
2,FRALLPGO000007,21,572.422
3,FRALLPGO000013,81,2514.619
4,FRALLPGO000014,11,427.522
...,...,...,...
2850,FRZUNP5724050077640808316,3,60.811
2851,FRZUNP5909246146811129817,2,86.666
2852,FRZUNP6746237616245941985,0,0.0
2853,FRZUNP6927750076048540479,0,0.0


### point de recharge

In [15]:
sampled_state_poc_g = get_sampled_state_poc(day, samples_per_day, sessions, statuses_s3)

In [16]:
#sampled_sessions[sampled_sessions[ID_POC] == 'FRA79E12346905331']
#sampled_statuses[sampled_statuses[ID_POC] == 'FRA79E12346905331'][100:150]
#sampled_state_poc_g[sampled_state_poc_g['pseudo_occupe'] >0]
#sampled_state_poc_g[sampled_state_poc_g[ID_POC] == 'FRA79E12346905331'][0:100]
#sessions_s3[sessions_s3[ID_POC] == 'FRA79E12346905331']
#statuses_s3[statuses_s3[ID_POC] == 'FRA79E12346905331']
#sessions_s3[sessions_s3[ID_POC] == 'FRHPCENF050462002']

In [17]:
state_poc_d_g = to_state_poc_d(sampled_state_poc_g, samples_per_day)

In [18]:
#full_state_poc_d_g = add_sessions_info(state_poc_d_g, sessions, ID_POC)
full_state_poc_d_g = pd.merge(state_poc_d_g, sessions_poc, on=ID_POC, how='left').fillna(0)

In [19]:
#state_poc_d_g[state_poc_d_g[ID_POC].str[:14] == 'FRHPCENF050462']
#state_poc_d_g[state_poc_d_g['pseudo_libre'] > 0]
#state_poc_d_g[state_poc_d_g['pseudo_occupe'] > 0]
full_state_poc_d_g[full_state_poc_d_g['sessions_nb'] > 30]

,id_pdc_itinerance,occupe,occupe_max,hors_service,libre,pseudo_libre,pseudo_occupe,sessions_nb,energy_cum
8675,FRELCE23WH,970.0,60.0,0.0,470.0,15.0,60.0,31,1069.004
9451,FRELCECFS8,1005.0,60.0,0.0,435.0,15.0,35.0,33,1007.832
9646,FRELCEEY8Q,1020.0,60.0,0.0,420.0,20.0,55.0,33,1124.864
9691,FRELCEFL8C,925.0,60.0,0.0,515.0,10.0,35.0,31,1006.441
9758,FRELCEGL52,745.0,60.0,20.0,675.0,10.0,25.0,32,890.59
9892,FRELCEJC6W,885.0,60.0,50.0,505.0,15.0,30.0,33,847.437
9907,FRELCEJJVC,955.0,60.0,0.0,485.0,20.0,60.0,33,1135.003
10015,FRELCEKWVV,910.0,60.0,0.0,530.0,20.0,35.0,31,961.139
10200,FRELCEN84H,885.0,60.0,0.0,555.0,15.0,45.0,31,929.87
10333,FRELCEPUMC,855.0,60.0,0.0,585.0,25.0,30.0,33,961.823


### station

In [20]:
sampled_state_station_g = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
print(len(sampled_state_station_g))

438336


In [21]:
sampled_state_station_g_pu = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO, add_full_use=True, add_latency=True)
print(len(sampled_state_station_g_pu))

438336


In [22]:
#sampled_state_station_g_pu

In [23]:
state_station_d_g_pu = to_state_grp_d(sampled_state_station_g_pu, ID_STATION, SAMPLES)
full_state_station_d_g_pu = pd.merge(state_station_d_g_pu, info_sessions_stations, on=ID_STATION, how='left').fillna(0)
full_state_station_d_g_pu

,id_station_itinerance,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len,sessions_nb,energy_cum
0,FRALDPFR00916,1,0.0,1425.0,20.0,15.0,0.0,0.0,15.0,20.0,0 days 00:20:00,1,56.7
1,FRALDPFR00950,1,0.0,1400.0,50.0,40.0,0.0,0.0,25.0,30.0,0 days 00:30:00,2,93.587
2,FRALLPGO000007,6,0.0,1010.0,0.0,0.0,0.0,430.0,0.0,0.0,0 days 00:00:00,21,572.422
3,FRALLPGO000013,10,0.0,520.0,0.0,0.0,0.0,920.0,0.0,0.0,0 days 00:00:00,81,2514.619
4,FRALLPGO000014,2,0.0,1140.0,75.0,65.0,0.0,235.0,40.0,45.0,0 days 00:45:00,11,427.522
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1517,FRZUNP2737850063890267871,2,0.0,1415.0,30.0,25.0,0.0,0.0,25.0,30.0,0 days 00:30:00,1,37.791
1518,FRZUNP2923250052026028059,1,285.0,1110.0,340.0,45.0,0.0,0.0,45.0,60.0,0 days 04:50:00,1,50.904
1519,FRZUNP5724050077640808316,3,0.0,1405.0,50.0,35.0,0.0,0.0,25.0,30.0,0 days 00:30:00,3,60.811
1520,FRZUNP5909246146811129817,1,0.0,1395.0,55.0,45.0,0.0,0.0,25.0,30.0,0 days 00:30:00,2,86.666


In [24]:

full_state_station_d_g_pu[(full_state_station_d_g_pu['sature_cum'] > 45) & (full_state_station_d_g_pu['nb_pdc'] > 5)]

,id_station_itinerance,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len,sessions_nb,energy_cum
215,FRAU1P00810000,9,0.0,770.0,150.0,150.0,520.0,0.0,60.0,60.0,0 days 02:30:00,2,0.2
272,FRDREP51172,6,0.0,1370.0,520.0,70.0,0.0,0.0,60.0,60.0,0 days 08:40:00,1,30.48
482,FREVCP000146,6,0.0,800.0,110.0,60.0,50.0,530.0,50.0,60.0,0 days 01:30:00,38,1234.259
483,FREVCP000147,6,0.0,575.0,180.0,85.0,130.0,650.0,45.0,60.0,0 days 01:30:00,52,1832.999
513,FREVCP000179,6,0.0,790.0,165.0,90.0,170.0,390.0,50.0,60.0,0 days 01:30:00,45,1595.879
532,FREVCP000199,6,0.0,500.0,340.0,140.0,195.0,605.0,45.0,60.0,0 days 01:40:00,69,2208.407
538,FREVCP000207,6,0.0,535.0,360.0,140.0,220.0,545.0,50.0,60.0,0 days 03:05:00,87,2911.099
545,FREVCP000214,6,0.0,830.0,135.0,55.0,90.0,465.0,35.0,60.0,0 days 01:00:00,43,1288.881
581,FREVCP000269,6,0.0,785.0,220.0,75.0,210.0,370.0,30.0,60.0,0 days 01:15:00,34,1168.368
582,FREVCP000282,6,0.0,420.0,130.0,50.0,85.0,885.0,30.0,45.0,0 days 00:40:00,65,2118.747


In [25]:
#e5_statics[e5_statics[ID_STATION] == 'FRPD1PBLDVDR']

In [28]:
#sessions_s3[sessions_s3[ID_POC] == 'FRPD1EBLDVDRKPC200015']

In [29]:
#statuses_s3[statuses_s3[ID_POC] == 'FRPD1EBLDVDRKPC200012']

In [30]:
sampled_state_station_g = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
print(len(sampled_state_station_g))

438336


In [32]:
def filter_sampled_state_poc(
    sampled_state_poc: pd.DataFrame, statics: pd.DataFrame
) -> pd.DataFrame:
    """Filter statuses and sessions with statics data."""
    filtered = sampled_state_poc[sampled_state_poc[ID_POC].isin(statics[ID_POC])].copy()
    return filtered

chunk_size = 200
codes, _ = pd.factorize(e5_statics[ID_STATION])
e5_statics["chunk"] = codes // chunk_size
chunks = e5_statics.groupby("chunk")
print(len(chunks))

futures = [
    to_sampled_state_grp(
        sampled_state_poc_g[sampled_state_poc_g[ID_POC].isin(chunk[ID_POC])],
        chunk,
        ID_STATION,
        SATURATION_RATIO,
        OVERLOAD_RATIO,
    )  # type: ignore[call-overload]
    for _, chunk in chunks
]

sampled_state_station_g = pd.concat(
    [future for future in futures], ignore_index=True
)
print(len(sampled_state_station_g))

15
438336


In [34]:
state_station_d_g = to_state_grp_d(sampled_state_station_g, ID_STATION, SAMPLES)

In [ ]:
#sampled_state_station_g
#sampled_state_station_g[sampled_state_station_g[ID_STATION] == 'FRHPCPNF080266TOTEM']
#state_station_h_g
#state_station_d_g
#state_station_d_g[state_station_d_g[ID_STATION] == 'FRHPCPNF080266TOTEM']
#state_station_d_g[state_station_d_g["sature_cum"] >= 120]

### Parc

In [35]:
parcs = e5_statics.rename(columns={ID_STATION: ID_POOL})
#parcs

In [36]:
sampled_state_parc_g = to_sampled_state_grp(sampled_state_poc_g, parcs, ID_POOL, SATURATION_RATIO, OVERLOAD_RATIO, add_full_use=True, add_latency=True)
print(len(sampled_state_parc_g))
sampled_state_parc_g

438336


,id_pool,periode,occupe,hors_service,libre,pleine_utilisation,pseudo_libre,pseudo_occupe,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
0,FRALDPFR00916,2026-07-14 00:00:00+00:00,0,0,1,0,0,0,1,False,True,False,False,False,False,2
1,FRALDPFR00916,2026-07-14 00:05:00+00:00,0,0,1,0,0,0,1,False,True,False,False,False,False,2
2,FRALDPFR00916,2026-07-14 00:10:00+00:00,0,0,1,0,0,0,1,False,True,False,False,False,False,2
3,FRALDPFR00916,2026-07-14 00:15:00+00:00,0,0,1,0,0,0,1,False,True,False,False,False,False,2
4,FRALDPFR00916,2026-07-14 00:20:00+00:00,0,0,1,0,0,0,1,False,True,False,False,False,False,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438331,FRZUNP7329346578064027187,2026-07-14 23:35:00+00:00,0,1,1,0,0,0,2,False,True,False,False,False,False,2
438332,FRZUNP7329346578064027187,2026-07-14 23:40:00+00:00,0,1,1,0,0,0,2,False,True,False,False,False,False,2
438333,FRZUNP7329346578064027187,2026-07-14 23:45:00+00:00,0,1,1,0,0,0,2,False,True,False,False,False,False,2
438334,FRZUNP7329346578064027187,2026-07-14 23:50:00+00:00,0,1,1,0,0,0,2,False,True,False,False,False,False,2


### divers

In [37]:
static = pd.DataFrame({'station' : ['s1']*3+['s2']*5+['s3']*3+['s4']*4+['s5']*5,
                      'pdc': ['p'+str(i) for i in range(20)]})
#static

In [38]:
chunk = 2
station_to_chunk = {
    station: i // chunk
    for i, station in enumerate(static["station"].drop_duplicates())
}

static["chunk"] = static["station"].map(station_to_chunk)
#for chunk_id, chunk in static.groupby("chunk", sort=True):
#    print(chunk_id)
#    print(chunk)
#static

In [39]:
chunk_size = 3
'''station_to_chunk = {
    station: i // chunk_size
    for i, station in enumerate(static["station"].drop_duplicates())
}

static["chunk"] = static["station"].map(station_to_chunk)
chunks = static.groupby("chunk", sort=True)
'''
codes, _ = pd.factorize(static["station"])
static["chunk"] = codes//chunk_size
chunks = static.groupby("chunk")
#for chunk_id, chunk in chunks:
    #print(chunk_id)
    #print(chunk)
futures = [
    chunk for _ , chunk in chunks
    #to_sampled_state_grp(sampled_state_poc, statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO) for _ , chunk in chunks
] 
static_new = pd.concat(
        [future for future in futures], ignore_index=True
    )
#static_new